# English → Spanish Transformer Translator

Trains a Transformer (from `rui.torch.transformer`) on `spa.txt` and saves
the complete inference pipeline so it can be loaded on any machine.

**Submission artefacts (written to `models/translation/`):**
| File | Contents |
|---|---|
| `transformer.pt` | Best model weights |
| `source_vectorizer.pkl` | English `TextVectorizer` |
| `target_vectorizer.pkl` | Spanish `TextVectorizer` |
| `config.json` | All hyperparameters |
| `training_curve.png` | Accuracy curve |

Run `inference.py` or `app.py` to translate on another machine without retraining.

## 0 · Setup — make the local `rui` package importable

In [ ]:
import os, sys

# Project root: the directory that contains this notebook and the rui/ folder
PROJECT_ROOT = os.path.abspath('')   # same as os.getcwd() inside a notebook cell
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)
print('rui package:', os.path.join(PROJECT_ROOT, 'rui'))

## 1 · Imports

In [ ]:
import json
import string
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from rui.utils import TextVectorizer
from rui.torch.transformer import Transformer
from rui.torch.utils import train, plotEpoch, ModelCheckpoint

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 2 · Hyperparameters

In [ ]:
# ── Assignment specification ──────────────────────────────────────────────
D_EMB      = 128
N_LAYERS   = 2
N_HEADS    = 8
D_FF       = 512
# ─────────────────────────────────────────────────────────────────────────
VOCAB_SIZE = 15_000
SEQ_LEN    = 20
BATCH_SIZE = 64
N_EPOCHS   = 15
LR         = 1e-3

DATA_FILE  = os.path.join(PROJECT_ROOT, 'spa.txt')
MODEL_DIR  = os.path.join(PROJECT_ROOT, 'models', 'translation')
os.makedirs(MODEL_DIR, exist_ok=True)
print('Model dir:', MODEL_DIR)

## 3 · Text standardisation
This function **must** be identical in both training and inference.

In [ ]:
_STRIP_CHARS = string.punctuation + '¿'
_STRIP_CHARS = _STRIP_CHARS.replace('[', '').replace(']', '')

def custom_standardization(text: str) -> str:
    text = text.lower()
    for ch in _STRIP_CHARS:
        text = text.replace(ch, '')
    return text

print(custom_standardization('Hello, World! ¿Cómo estás?'))

## 4 · Load & split data

In [ ]:
print(f'Loading {DATA_FILE} ...')
with open(DATA_FILE, encoding='utf-8') as f:
    lines = f.read().split('\n')[:-1]

text_pairs = []
for line in lines:
    parts = line.split('\t')
    if len(parts) < 2:
        continue
    english, spanish = parts[0], parts[1]
    spanish = '[start] ' + spanish + ' [end]'
    text_pairs.append((english, spanish))

random.seed(42)
random.shuffle(text_pairs)

num_val   = int(0.15 * len(text_pairs))
num_train = len(text_pairs) - 2 * num_val
train_pairs = text_pairs[:num_train]
val_pairs   = text_pairs[num_train : num_train + num_val]
test_pairs  = text_pairs[num_train + num_val :]

print(f'Total pairs: {len(text_pairs):,}')
print(f'Train: {len(train_pairs):,}  Val: {len(val_pairs):,}  Test: {len(test_pairs):,}')
print('Example pair:', text_pairs[0])

## 5 · Build & save TextVectorizers
We save them **immediately** so they are available on another machine.

In [ ]:
source_vectorizer = TextVectorizer(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=SEQ_LEN,
    standardize=custom_standardization,
)
source_vectorizer.adapt([p[0] for p in train_pairs])

# +1 so spa[:,:-1] and spa[:,1:] each have length SEQ_LEN
target_vectorizer = TextVectorizer(
    max_tokens=VOCAB_SIZE,
    output_sequence_length=SEQ_LEN + 1,
    standardize=custom_standardization,
)
target_vectorizer.adapt([p[1] for p in train_pairs])

src_vocab_size = len(source_vectorizer.get_vocabulary())
tgt_vocab_size = len(target_vectorizer.get_vocabulary())
start_idx = target_vectorizer.word_to_idx.get('[start]')
end_idx   = target_vectorizer.word_to_idx.get('[end]')

print(f'Src vocab: {src_vocab_size}  Tgt vocab: {tgt_vocab_size}')
print(f'[start] idx: {start_idx}  [end] idx: {end_idx}')

# ── Save vectorizers now ──────────────────────────────────────────────────
source_vectorizer.save(os.path.join(MODEL_DIR, 'source_vectorizer.pkl'))
target_vectorizer.save(os.path.join(MODEL_DIR, 'target_vectorizer.pkl'))
print('Vectorizers saved.')

## 6 · Save config (so inference can rebuild the model architecture)

In [ ]:
config = {
    'd_emb':          D_EMB,
    'n_layers':       N_LAYERS,
    'n_heads':        N_HEADS,
    'd_ff':           D_FF,
    'seq_len':        SEQ_LEN,
    'src_vocab_size': src_vocab_size,
    'tgt_vocab_size': tgt_vocab_size,
    'start_idx':      start_idx,
    'end_idx':        end_idx,
}
with open(os.path.join(MODEL_DIR, 'config.json'), 'w') as f:
    json.dump(config, f, indent=2)
print('config.json saved:', config)

## 7 · Dataset & DataLoaders

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vec, tgt_vec):
        self.pairs   = pairs
        self.src_vec = src_vec
        self.tgt_vec = tgt_vec

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, i):
        eng, spa = self.pairs[i]
        eng_vec = torch.tensor(self.src_vec(eng)[0], dtype=torch.long)
        spa_vec = torch.tensor(self.tgt_vec(spa)[0], dtype=torch.long)
        return eng_vec, spa_vec[:-1], spa_vec[1:]   # (src, tgt_in, tgt_out)

train_loader = DataLoader(
    TranslationDataset(train_pairs, source_vectorizer, target_vectorizer),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
)
val_loader = DataLoader(
    TranslationDataset(val_pairs, source_vectorizer, target_vectorizer),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
)

# Quick shape check
eng_b, tgt_in_b, tgt_out_b = next(iter(train_loader))
print(f'eng: {eng_b.shape}  tgt_in: {tgt_in_b.shape}  tgt_out: {tgt_out_b.shape}')

## 8 · Loss & accuracy helpers

In [ ]:
def masked_loss(pred, label):
    """Cross-entropy that ignores padding tokens (index 0)."""
    loss_fn = nn.CrossEntropyLoss(reduction='none')
    pred    = pred.view(-1, pred.size(-1))   # (B*T, V)
    label   = label.view(-1)                 # (B*T,)
    loss    = loss_fn(pred, label)
    mask    = (label != 0).float()
    return (loss * mask).sum() / mask.sum()

def masked_accuracy(pred, label):
    pred  = torch.argmax(pred, dim=2)        # (B, T)
    mask  = label != 0
    match = (label == pred) & mask
    return match.float().sum() / mask.float().sum()

## 9 · Build Transformer model

In [ ]:
model = Transformer(
    n_layers       = N_LAYERS,
    d_emb          = D_EMB,
    n_heads        = N_HEADS,
    d_ff           = D_FF,
    src_vocab_size = src_vocab_size,
    tgt_vocab_size = tgt_vocab_size,
    seq_len        = SEQ_LEN,
    dropout        = 0.1,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## 10 · Train

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LR)
# Static LR schedule (dummy, keeps the rui.torch.utils API happy)
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: 1.0)

# Saves the best checkpoint automatically during training
checkpoint = ModelCheckpoint(
    filepath=os.path.join(MODEL_DIR, 'transformer.pt'),
    monitor='val_loss',
    save_best_only=True,
)

history = train(
    model, train_loader, val_loader, optimizer,
    loss_fn         = masked_loss,
    scheduler       = scheduler,
    accuracy_fn     = masked_accuracy,
    callbacks       = [checkpoint],
    device          = device,
    n_batch_per_report = 200,
    n_epochs        = N_EPOCHS,
)

## 11 · Training curve

In [ ]:
import matplotlib.pyplot as plt

epochs = range(len(history['train_accuracy']))
plt.figure(figsize=(9, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history['average_train_loss'], 'k', label='Train')
plt.plot(epochs, history['average_val_loss'],   'b', label='Val')
plt.axvline(x=int(min(range(len(history['average_val_loss'])),
                       key=history['average_val_loss'].__getitem__)), color='0.5')
plt.title('Average Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history['train_accuracy'], 'k', label='Train')
plt.plot(epochs, history['val_accuracy'],   'b', label='Val')
plt.axvline(x=int(max(range(len(history['val_accuracy'])),
                       key=history['val_accuracy'].__getitem__)), color='0.5')
plt.title('Masked Accuracy (%)')
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'training_curve.png'), dpi=120)
plt.show()
print('training_curve.png saved.')

## 12 · Confirm saved artefacts

In [ ]:
for fname in ['transformer.pt', 'source_vectorizer.pkl',
              'target_vectorizer.pkl', 'config.json', 'training_curve.png']:
    path = os.path.join(MODEL_DIR, fname)
    size = os.path.getsize(path) / 1024
    print(f'  ✓  {fname:<30}  {size:>8.1f} KB')

## 13 · Inference demo

The cell below loads everything from disk — simulating a fresh machine with **no** training state in memory.

In [ ]:
# ── Load pipeline from disk ───────────────────────────────────────────────
import json
from rui.utils import TextVectorizer
from rui.torch.transformer import Transformer

def _standardize(text):
    """Must be identical to the standardizer used during training."""
    import string
    strip = string.punctuation + '¿'
    strip = strip.replace('[','').replace(']','')
    text  = text.lower()
    for ch in strip:
        text = text.replace(ch, '')
    return text

def load_translator(model_dir):
    dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    with open(os.path.join(model_dir, 'config.json')) as f:
        cfg = json.load(f)
    src_vec = TextVectorizer.load(
        os.path.join(model_dir, 'source_vectorizer.pkl'),
        standardize=_standardize)
    tgt_vec = TextVectorizer.load(
        os.path.join(model_dir, 'target_vectorizer.pkl'),
        standardize=_standardize)
    mdl = Transformer(
        n_layers=cfg['n_layers'], d_emb=cfg['d_emb'],
        n_heads=cfg['n_heads'],   d_ff=cfg['d_ff'],
        src_vocab_size=cfg['src_vocab_size'],
        tgt_vocab_size=cfg['tgt_vocab_size'],
        seq_len=cfg['seq_len'],   dropout=0.0,
    )
    state = torch.load(os.path.join(model_dir, 'transformer.pt'),
                       map_location=dev, weights_only=True)
    mdl.load_state_dict(state)
    mdl.to(dev).eval()
    return mdl, src_vec, tgt_vec, cfg, dev


def translate(sentence, mdl, src_vec, tgt_vec, cfg, dev):
    """Greedy decode a single English sentence to Spanish."""
    seq_len   = cfg['seq_len']
    start_idx = cfg['start_idx']
    end_idx   = cfg['end_idx']

    src_ids = torch.tensor(src_vec(sentence), dtype=torch.long).to(dev)  # (1, seq_len)
    decoded = [start_idx]

    with torch.no_grad():
        for _ in range(seq_len):
            tgt_ids = decoded[-seq_len:]
            tgt_ids = tgt_ids + [0] * (seq_len - len(tgt_ids))       # right-pad
            tgt_t   = torch.tensor([tgt_ids], dtype=torch.long).to(dev)
            logits  = mdl((src_ids, tgt_t))                           # (1, seq_len, V)
            pos     = min(len(decoded) - 1, seq_len - 1)
            nxt     = torch.argmax(logits[0, pos, :]).item()
            if nxt == end_idx or nxt == 0:
                break
            decoded.append(nxt)

    words = [tgt_vec.idx_to_word.get(i, '') for i in decoded[1:]]
    words = [w for w in words if w and w not in ('[end]', '[start]')]
    return ' '.join(words) or '(no translation)'


# Load from saved files (zero training-time state needed)
inf_model, inf_src_vec, inf_tgt_vec, inf_cfg, inf_dev = load_translator(MODEL_DIR)
print('Model loaded from disk. Ready.')

In [ ]:
test_sentences = [
    'Hello, how are you?',
    'I love learning new languages.',
    'Where is the nearest hospital?',
    'The weather is beautiful today.',
    'What time does the train leave?',
    'Thank you very much for your help.',
    'I would like a table for two, please.',
    'Can you speak more slowly?',
]

print(f'{"English":<45}  {"Spanish"}')
print('-' * 80)
for sent in test_sentences:
    spa = translate(sent, inf_model, inf_src_vec, inf_tgt_vec, inf_cfg, inf_dev)
    print(f'{sent:<45}  {spa}')

## 14 · Interactive translation

Type any English sentence to see the translation.

In [ ]:
while True:
    try:
        sentence = input('English > ').strip()
    except EOFError:
        break
    if not sentence or sentence.lower() in ('quit', 'exit', 'q'):
        print('Goodbye!')
        break
    result = translate(sentence, inf_model, inf_src_vec, inf_tgt_vec, inf_cfg, inf_dev)
    print(f'Spanish > {result}\n')